# Colab Regression Worker

Run this notebook in a Colab GPU runtime. The first code cell now loads `GITHUB_TOKEN` from Colab Secrets when available, or asks you to paste it into a runtime input prompt when using the VS Code Colab extension. The token is not written to the notebook file. The first bring-up loop runs a safe backend check for 5 cycles, pulls latest GitHub code each cycle, and pushes worker manifests/logs back to GitHub.

## 1. Clone or Update Repository

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/DrYGuo/Aberration-Simulation.git"
REPO_DIR = Path("/content/Aberration-Simulation")

def load_github_token():
    # 1. Colab web UI Secrets, if available.
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
        if token:
            return token.strip(), "Colab Secrets"
    except Exception:
        pass

    # 2. Runtime environment, if already set by another cell.
    token = os.environ.get("GITHUB_TOKEN")
    if token:
        return token.strip(), "runtime environment"

    # 3. VS Code Colab extension fallback. Paste into the input prompt, not into code.
    print("Paste your fine-grained GitHub token into the input box for this cell.")
    print("It is used only for this Colab runtime and is not written to the notebook file.")
    token = input("GITHUB_TOKEN: ").strip()
    if token:
        return token, "manual notebook input"
    return "", "missing"

token, token_source = load_github_token()
if not token:
    raise RuntimeError("GITHUB_TOKEN was not provided; cannot push worker artifacts to GitHub.")
os.environ["GITHUB_TOKEN"] = token
print(f"GITHUB_TOKEN loaded from {token_source}.")

if REPO_DIR.exists():
    subprocess.run(["git", "fetch", "origin", "main"], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", "main"], cwd=REPO_DIR, check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print("repo:", REPO_DIR)
print("commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())


## 2. Check GPU

In [ ]:
!nvidia-smi
!python scripts/check_backend.py

## 3. Start 5-Cycle Worker Loop

In [ ]:
!python scripts/colab_regression_worker.py --config experiments/colab_worker_5cycles.json --cycles 5